In [ ]:
"""
Data analysis functions
"""

import pandas as pd
import numpy as np
from config import settings


def analyze_lab_test_completeness(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze completeness of laboratory tests.
    
    Returns:
        DataFrame with completeness statistics for each lab test
    """
    any_list, all_list, initial_list = [], [], []
    missing_counts, missing_percentages = [], []
    
    for test in settings.LAB_TESTS:
        # Patients with at least one measurement
        any_count = df.groupby('ID2')[test].apply(lambda x: x.notnull().any()).sum()
        any_list.append(any_count)
        
        # Patients with data at all time points
        all_count = df.groupby('ID2')[test].apply(lambda x: x.notnull().all()).sum()
        all_list.append(all_count)
        
        # Patients with data at initial time point
        first_time = df.groupby('ID2')['Time'].min().reset_index()
        first_data = df.merge(first_time, on=['ID2', 'Time'])
        initial_count = first_data[test].notnull().sum()
        initial_list.append(initial_count)
        
        # Missing values
        missing_count = df[test].isnull().sum()
        missing_counts.append(missing_count)
        missing_percentage = (missing_count / len(df) * 100).round(2)
        missing_percentages.append(missing_percentage)
    
    # Build results table
    table_s1 = pd.DataFrame({
        'Laboratory tests': settings.LAB_TESTS,
        'Normal ranges': [''] * len(settings.LAB_TESTS),
        'Total patients with ≥1 measurement': any_list,
        'Patients with data at all time points': all_list,
        'Patients with data at initial time point': initial_list,
        'Total measurements (time points)': [len(df)] * len(settings.LAB_TESTS),
        'Measurements available': df[settings.LAB_TESTS].notnull().sum().values,
        'Missing values (count)': missing_counts,
        'Missing values (%)': missing_percentages
    })
    
    # Calculate percentage
    table_s1['Percentage (%)'] = (
        table_s1['Measurements available'] / 
        table_s1['Total measurements (time points)'] * 100
    ).round(2)
    
    return table_s1


def analyze_time_points(df: pd.DataFrame) -> dict:
    """Analyze time points distribution per patient."""
    df = df.copy()
    df['ID2_str'] = df['ID2'].astype(str)
    patient_counts = df['ID2_str'].value_counts()
    
    summary = {
        'average': patient_counts.mean(),
        'median': patient_counts.median(),
        'std': patient_counts.std(),
        'min': patient_counts.min(),
        'max': patient_counts.max(),
        'Q1': patient_counts.quantile(0.25),
        'Q3': patient_counts.quantile(0.75)
    }
    
    return summary


def analyze_time_intervals(df: pd.DataFrame) -> dict:
    """Analyze time intervals between consecutive measurements."""
    df_sorted = df.sort_values(['ID2', 'Time']).copy()
    df_sorted['interval_hours'] = df_sorted.groupby('ID2')['Time'].diff().dt.total_seconds() / 3600
    all_intervals = df_sorted['interval_hours'].dropna()
    
    summary = {
        'total_intervals': len(all_intervals),
        'min_hours': all_intervals.min(),
        'max_days': all_intervals.max() / 24,
        'median_hours': all_intervals.median(),
        'mean_hours': all_intervals.mean(),
        'IQR': [all_intervals.quantile(0.25), all_intervals.quantile(0.75)]
    }
    
    return summary